# Degerlendirme notebook'u: `qwen25vl_7b`

- Model: `Qwen/Qwen2.5-VL-7B-Instruct`
- **TEMIZ runtime** gerekir (unsloth ile ayni ortama KURULMAZ).
- Motor: vLLM (sorun cikarsa `ENGINE="hf"`).
- Ciktilar: Drive'da `kap_lora/eval/qwen25vl_7b/` altina yazilir.


In [ ]:
# Ayarlar
MODEL_KEY = "qwen25vl_7b"
DRIVE_DIR = "/content/drive/MyDrive/kap_lora"
ENGINE = "vllm"
MAX_NEW_TOKENS = 3072
GPU_MEM = 0.90
MAX_NUM_SEQS = 128
DATASET_ZIP = f"{DRIVE_DIR}/dataset.zip"
REPO_URL = "github.com/furkancmc/finance-vl-fine-tune"


In [ ]:
# Kurulum: Drive, paketler, HF token
from google.colab import drive
drive.mount("/content/drive")
!pip install -q vllm rapidfuzz qwen-vl-utils
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        from huggingface_hub import login
        login(tok)
        print("HF login ok")
except Exception as e:
    print("HF token yok/atlandi:", e)


In [ ]:
# Repo cek
import os, shutil, subprocess
from pathlib import Path
gh = None
try:
    from google.colab import userdata
    gh = userdata.get("GH_TOKEN")
except Exception:
    pass
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
if gh:
    url = f"https://{gh}@{REPO_URL}.git"
    subprocess.run(["git", "clone", url, "/content/repo"], check=True)
else:
    src = Path(DRIVE_DIR) / "code"
    assert src.exists(), f"GH_TOKEN yok ve {src} bulunamadi"
    shutil.copytree(src, "/content/repo")
%cd /content/repo


In [ ]:
# Veri: dataset.zip ac
import os, zipfile
if not os.path.exists("/content/dataset"):
    assert os.path.exists(DATASET_ZIP), f"zip yok: {DATASET_ZIP}"
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall("/content")
print("dataset hazir")
import transformers
from packaging.version import Version
from model_zoo import get
need = get(MODEL_KEY)["min_transformers"]
print("transformers", transformers.__version__, "gerekli >=", need)
assert Version(transformers.__version__) >= Version(need), "transformers cok eski: !pip install -U transformers"


## Baz model degerlendirmesi

In [ ]:
!python eval_vllm.py --model $MODEL_KEY --tag base \
  --data /content/dataset \
  --engine $ENGINE --max-new-tokens $MAX_NEW_TOKENS \
  --gpu-mem $GPU_MEM --max-num-seqs $MAX_NUM_SEQS \
  --out {DRIVE_DIR}/eval/qwen25vl_7b


## LoRA (birlestirilmis) degerlendirmesi

In [ ]:
!python eval_vllm.py --model $MODEL_KEY --tag lora \
  --model-path {DRIVE_DIR}/qwen25vl_7b/merged \
  --data /content/dataset \
  --engine $ENGINE --max-new-tokens $MAX_NEW_TOKENS \
  --gpu-mem $GPU_MEM --max-num-seqs $MAX_NUM_SEQS \
  --out {DRIVE_DIR}/eval/qwen25vl_7b


## Rapor

In [ ]:
!python report.py --eval-dir {DRIVE_DIR}/eval/qwen25vl_7b
from IPython.display import Markdown, display
from pathlib import Path
p = Path(f"{DRIVE_DIR}/eval/qwen25vl_7b/report.md")
if p.exists():
    display(Markdown(p.read_text(encoding="utf-8")))
else:
    print("report.md bulunamadi")


## Not

vLLM sorun cikarirsa ayar hucresinde `ENGINE = "hf"` yapip yeniden calistirin.
